---

digital brain MEG pipeline

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2

# load pgl actions
from pgl import pglActions

---

load session

---

In [ ]:

# Load the session
#loadSession = pglActions.loadSession()
#loadSession.configure()
#if loadSession: session = loadSession.run()

---

Load FieldLine data

---

In [ ]:
# execute load
session = pglActions.loadFieldline.execute()

(loadFieldline:_run) Loading Fieldline FIF file:
/Users/justin/data/things/s0004/session_2026-08-25_meg-opm/20260825_105837_sub-S004_file-ThingsPilotRun09_raw.fif
(loadFieldline:_run) Loading Fieldline FIF file:
/Users/justin/data/things/s0004/session_2026-08-25_meg-opm/20260825_105019_sub-S004_file-ThingsPilotRun07_raw.fif
(loadFieldline:_run) Loading Fieldline FIF file:
/Users/justin/data/things/s0004/session_2026-08-25_meg-opm/sub-S004_file-ThingsPilotRun01_raw.fif
(loadFieldline:_run) Loading Fieldline FIF file:
/Users/justin/data/things/s0004/session_2026-08-25_meg-opm/20260825_103412_sub-S004_file-ThingsPilotRun03_raw.fif
(loadFieldline:_run) Loading Fieldline FIF file:
/Users/justin/data/things/s0004/session_2026-08-25_meg-opm/20260825_102956_sub-S004_file-ThingsPilotRun02_raw.fif
(loadFieldline:_run) Loading Fieldline FIF file:
/Users/justin/data/things/s0004/session_2026-08-25_meg-opm/20260825_104244_sub-S004_file-ThingsPilotRun05_raw.fif
(loadFieldline:_run) Loading Fieldline

---

Concatenate raws

---

In [ ]:
# configure the concatenation
mneConcatenate = pglActions.mneConcatenate()
mneConcatenate.configure(session, all=True)

In [ ]:
# run the concatenation
newSession = mneConcatenate.run(session)
if newSession: session = newSession

---

Set names of events

---

In [ ]:
# configure the concatenation
mneConfigureEvents = pglActions.mneConfigureEvents()

# group all the things labels into one
grouped = {
    'things': range(2,201),
    'blank': 1022,
    'catch': 1023
}

# label each stimulus with its own name
each = {
    f"stimulus {code:03d}": code
    for code in range(2, 201)
}
each["blank"] = 1022
each["catch"] = 1023

triggerLabelSets = {
    "grouped": grouped,
    "each": each
}

# configure
mneConfigureEvents.configure(triggerChannel='di2', triggerShortestEvent=1, triggerLabelSets = triggerLabelSets)

In [ ]:
# configure the events
newSession = mneConfigureEvents.run(session)
if newSession: session = newSession

---

Filter data

---

In [ ]:
mneFilter = pglActions.mneFilter()
mneFilter.configure(lowCutoff=0.5, highCutoff=120.0, notch=True, notchFrequency=60.0)

In [ ]:
newSession = mneFilter.run(session)
if newSession: session = newSession

---

Bads handling

---

In [ ]:
mneBads = pglActions.mneBads()
mneBads.configure(excludeWhenPositionIsNaN=True, extraBads=[], interpolationOrigin="zero", method="interpolate")

In [ ]:
newSession = mneBads.run(session)
if newSession: session = newSession

---

create epochs

---

In [ ]:
mneCreateEpochs = pglActions.mneCreateEpochs()
mneCreateEpochs.configure(tmin=0.0, tmax=2.0)

In [ ]:
newSession = mneCreateEpochs.run(session)
if newSession: session = newSession

---

Plot evoked

---

In [ ]:
mnePlotEvoked = pglActions.mnePlotEvoked()
#mnePlotEvoked.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks="L213")
#mnePlotEvoked.configure(set="each",label="stimulus 010")
mnePlotEvoked.configure(set="grouped",label="catch",minFreq=0,maxFreq=20)

In [ ]:
newSession = mnePlotEvoked.run(session)
if newSession: session = newSession

---

Downsample epochs

---

In [ ]:
mneDownsampleEvoked = pglActions.mneDownsampleEvoked()
mneDownsampleEvoked.configure(downsampleFrequency=200)

In [ ]:
newSession = mneDownsampleEvoked.run(session)
if newSession: session = newSession

---

Compute NCSNR

---

In [ ]:
mneComputeNCSNR = pglActions.mneComputeNCSNR()
#mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag",channelName="L213_bz-s115")
#mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag")
mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag",channelName="L213_bz-s115")


In [ ]:
newSession = mneComputeNCSNR.run(session)
if newSession: session = newSession

---

look at spectrum

---

---

History

---

In [ ]:
session.history()

---

SCRATCH code

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pathEffects

channelName, peakTime, peakAmplitude = session.mne.evoked.get_peak(ch_type="mag", mode="abs", return_amplitude=True)
peakTime=0.250
with plt.rc_context({"figure.constrained_layout.use": False, "figure.autolayout": False}):
    fig = session.mne.evoked.plot_topomap(
        times=peakTime,
        ch_type="mag",
        average=0.02,
        sensors=True,
        show_names=True,
        show=False,
    )

    fig.set_size_inches(24, 16, forward=True)

    # ax.texts contains the channel-name labels added to the topomap.
    for ax in fig.axes:
        for text in ax.texts:
            text.set_color("white")
            text.set_fontsize(12)
            text.set_fontweight("bold")
            text.set_path_effects([
                pathEffects.Stroke(linewidth=2.5, foreground="black"),
                pathEffects.Normal(),
            ])

    fig.suptitle(f"Topomap at peak: {channelName} | {peakTime * 1000:.1f} ms", fontsize=18, fontweight="bold")
    plt.show()
print(session.mne.evoked.ch_names)

In [ ]:
session.mne.lookupSensor('L101')

Do spectrum on the evoked averages - take the mean of the 3000 trials in time domain and transform last
do it over 2 seconds - you should see nothing at 0.5Hz 

Compute how many seconds (in averages) and how many trials
Same yaxis
to look at sqrt(n) - compute 
Deflate the noise to compare to the number of trials

In [ ]:
session.mne.raw.ch_names